# 🛠️ Section 2 — NumPy Avancé : Matrices 2D, Agrégations Multi-Axes, Broadcasting et Aléatoire

<div align="center" style="margin: 20px 0 25px 0;">
  <img src="pictures/cartoon_section2_cover.jpg" alt="NumPy Avancé : The Matrix Odyssey" width="88%" style="border-radius: 12px; box-shadow: 0 4px 25px rgba(0,0,0,0.4);" />
  <br>
  <em style="color: #94a3b8; font-size: 0.9em;">🚀 Bienvenue dans la Matrix Odyssey : domptez les tenseurs 2D, l'écrasement multi-axes, les clones holographiques du broadcasting et le lissage de signaux !</em>
</div>

> 🎬 **Ce que raconte cette affiche d'ouverture (*The Matrix Odyssey*) :**  
> Dans la Section 1, vous avez voyagé sur une seule ligne droite : le vecteur 1D de 100 000 mesures de puissance.  
> Ici, nous franchissons le pas de l'**espace multidimensionnel** :  
> 1. **La Grille 2D & Tenseurs** : organiser les données en lignes (individus / jours) et colonnes (variables / heures).  
> 2. **Les Deux Forces Cosmiques d'`axis`** : la gravité verticale `axis=0` (qui compresse les colonnes) et la tornade horizontale `axis=1` (qui compresse les lignes).  
> 3. **Le Moteur de Broadcasting** : projeter des hologrammes virtuels pour faire des calculs matriciels complexes sans consommer le moindre mégaoctet supplémentaire en RAM.  
> 4. **Le Robot Croupier `default_rng`** : maîtriser le hasard et garantir des simulations 100% reproductibles grâce à la clé dorée `seed=42`.  
> 5. **Le Rouleau de Lissage `np.convolve`** : nettoyer les signaux électriques bruités par fenêtre glissante.

---



Bienvenue dans le second module d'outils du cours **Data Science & Python** !

Dans la **Section 1**, nous avons posé les fondations : comment un vecteur 1D de 100 000 mesures de puissance électrique est stocké dans un bloc contigu de mémoire vive (RAM), comment éliminer les boucles `for` grâce à la vectorisation matérielle (**SIMD**), et comment auditer des dépassements de seuil sans déréférencer le moindre pointeur.

---

### 🌟 Du signal 1D au tableau 2D : Pourquoi changer de dimension ?
Dans le monde réel de l'Internet des Objets (IoT) et de la Data Science, les données arrivent rarement sous la forme d'un simple signal isolé. Elles s'organisent naturellement selon **plusieurs dimensions** :
1. **La dimension temporelle périodique** : Un relevé continu de capteur Linky (par exemple 720 mesures horaires sur 30 jours) n'est qu'un long ruban abstrait en 1D. En le repliant en une matrice de **30 jours × 24 heures**, chaque ligne devient une journée complète et chaque colonne une tranche horaire. On peut alors instantanément comparer le lundi au dimanche, ou superposer les pics de soirée !
2. **La dimension multi-capteurs** : Une maison connectée ne mesure pas un seul appareil, mais plusieurs circuits simultanément (cuisine, salon, pompe à chaleur, recharge de voiture). Chaque instant temporel est un vecteur de mesures, et l'ensemble forme une **matrice d'observations**.
3. **Le format universel du Machine Learning** : Comme nous le verrons dès la Section 3 (Perceptron) et dans les projets *Energy Lab*, toutes les bibliothèques d'IA (**Scikit-Learn**, **PyTorch**) exigent des données sous la forme d'une matrice $X$ de taille $(N, M)$ :
   - $N$ **lignes** : les échantillons ou observations (*samples*).
   - $M$ **colonnes** : les variables explicatives (*features*).

---

### 🎯 Objectifs opérationnels de ce notebook :
1. **Penser en 2D** : Maîtriser l'indexation `[ligne, colonne]` et le remodelage de dimension `.reshape()` sans duplication mémoire.
2. **Comprendre la mémoire 2D** : Découvrir pourquoi une matrice 2D n'est qu'un ruban 1D en mémoire physique (*Row-Major / C-Order*) et pourquoi les opérations sont instantanées.
3. **Maîtriser le paramètre `axis`** : Éliminer définitivement la confusion entre `axis=0` (réduction verticale par colonne) et `axis=1` (réduction horizontale par ligne).
4. **Exploiter le Broadcasting** : Réaliser des calculs entre matrices et vecteurs de dimensions différentes (standardisation Z-Score) sans boucle.
5. **Simuler des distributions réalistes** : Utiliser le générateur aléatoire moderne `np.random.default_rng` pour injecter du bruit thermique et des anomalies.
6. **Filtrer un signal par convolution** : Lisser une courbe de puissance avec `np.convolve` (moyenne mobile vectorisée).

> 💡 **Pédagogie interactive** : Ce notebook contient des **QCM interactifs** (dont le code est masqué par défaut) et des cellules de vérification automatique `check()` pour valider vos réponses en direct.  
> 👉 `Shift + Entrée` pour exécuter chaque cellule pas à pas.

In [ ]:
# Programme 1 — Initialisation & Outils interactifs (QCM et validation)
import numpy as np
import matplotlib.pyplot as plt
import random as _random_qcm
import ipywidgets as w
from IPython.display import display

# Relances humoristiques adaptées
_RELANCES_HUMOUR = [
    "Erreur sur l'axe : tu as confondu axis=0 et axis=1 !",
    "Erreur de dimension (BroadcastError) dans ton raisonnement !",
    "Attention à la forme : vérifie bien avec .shape !",
    "Pas d'inquiétude, même les créateurs de Scikit-Learn ont dû relire la doc de reshape un jour.",
    "Retente ta chance, le tableau 2D t'attend !"
]

def qcm(question, options, correct, explication=""):
    """
    Affiche un QCM interactif avec ipywidgets.
    correct : index (int) ou liste d'index de la/des bonne(s) réponse(s).
    """
    if isinstance(correct, int):
        correct = [correct]

    boutons = w.RadioButtons(options=options, layout=w.Layout(width='100%'))
    sortie  = w.Output()
    titre   = w.HTML(f"<b>❓ {question}</b>")

    def verifier(change):
        idx = options.index(change['new'])
        sortie.clear_output()
        with sortie:
            if idx in correct:
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#d4edda;color:#155724;border-left:5px solid #28a745'>"
                    f"✅ <b>Bravo, c'est exact !</b><br>{explication}</div>"
                ))
            else:
                humour = _random_qcm.choice(_RELANCES_HUMOUR)
                display(w.HTML(
                    f"<div style='padding:10px;border-radius:8px;background:#f8d7da;color:#721c24;border-left:5px solid #dc3545'>"
                    f"❌ <b>Pas tout à fait...</b> {humour}<br><br>"
                    f"<b>Explication :</b> {explication}</div>"
                ))

    boutons.observe(verifier, names='value')
    display(w.VBox([titre, boutons, sortie]))

def check(nom, valeur, attendu, tol=1e-5):
    """
    Vérifie la valeur d'une variable ou d'un array calculé dans un exercice.
    Affiche un retour visuel vert ou rouge.
    """
    ok = False
    try:
        if isinstance(attendu, np.ndarray) and isinstance(valeur, np.ndarray):
            ok = valeur.shape == attendu.shape and np.allclose(valeur, attendu, atol=tol)
        elif isinstance(attendu, (int, float, np.integer, np.floating)) and isinstance(valeur, (int, float, np.integer, np.floating)):
            ok = abs(valeur - attendu) <= tol
        elif isinstance(attendu, (tuple, list)) and isinstance(valeur, (tuple, list)):
            ok = len(valeur) == len(attendu) and all(abs(v - a) <= tol for v, a in zip(valeur, attendu))
        else:
            ok = bool(valeur == attendu)
    except Exception:
        ok = False

    if ok:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#d4edda;color:#155724;border-left:4px solid #28a745'>"
            f"✅ <b>{nom}</b> est correct ! (Valeur : <code>{repr(valeur)}</code>)</div>"
        ))
    else:
        display(w.HTML(
            f"<div style='padding:8px 12px;border-radius:6px;background:#f8d7da;color:#721c24;border-left:4px solid #dc3545'>"
            f"❌ <b>{nom}</b> n'est pas encore la valeur attendue.<br>"
            f"Ta valeur : <code>{repr(valeur)}</code><br>"
            f"Valeur attendue : <code>{repr(attendu)}</code></div>"
        ))

print(f"✅ Environnement initialisé avec NumPy version {np.__version__}")
print("✅ Fonctions qcm() et check() prêtes à l'emploi.")

---
## 1 · Tableaux 2D (Matrices) et Réorganisation (`reshape`)

<div align="center" style="margin: 15px 0 25px 0;">
  <img src="pictures/cartoon_s2_ch1_matrices_reshape.jpg" alt="Matrices 2D et Reshape" width="85%" style="border-radius: 10px; box-shadow: 0 4px 20px rgba(0,0,0,0.35);" />
  <br>
  <em style="color: #94a3b8; font-size: 0.9em;">👨‍🍳 Le chef plie le long spaghetti 1D en plat de lasagnes 3×4 avec <code>reshape(3, 4)</code> : zéro copie en RAM, c'est la même mémoire vue sous un autre angle !</em>
</div>

> 💡 **Ce que nous enseigne la métaphore du chef cuisinier :**  
> - **La barrette de RAM ne connaît pas la 2D** : physiquement, la mémoire vive d'un ordinateur est un long ruban linéaire d'adresses d'octets consécutifs (notre spaghetti 1D).  
> - **Le moule à lasagnes 3×4 (`reshape`)** : créer une matrice 2D ne crée *aucune* nouvelle donnée en mémoire ($O(1)$). NumPy se contente de modifier les **métadonnées** (`shape=(3, 4)` et `strides`), c'est-à-dire la règle de saut d'index pour se déplacer de ligne en ligne.  
> - **Les lunettes `ravel()`** : à tout instant, réaplatir la matrice en vecteur 1D est gratuit et immédiat car les nombres sont déjà rangés les uns à côté des autres en mémoire !



En Data Science et en ingénierie des données, le tableau à 2 dimensions (la matrice) est la structure reine.  
Mais avant de manipuler des indices, il est crucial de comprendre comment la 2D est pensée en modélisation et comment elle est gérée par le matériel.

---

### 📊 1.1 Du signal continu au format tabulaire : La double lecture

Dans notre laboratoire d'analyse énergétique, une matrice 2D s'interprète selon deux visions complémentaires :

1. **La vision « Séries Temporelles » (Périodicité Jours × Créneaux horaires)** :
   - Chaque **ligne** $i$ représente une **période complète** (ex. le Jour $i$).
   - Chaque **colonne** $j$ représente un **créneau récurrent** (ex. la mesure prise au créneau de 12h00).
   - *Intérêt analytique* : Isoler la ligne `mat[0, :]` donne les températures de toute la première journée. Isoler la colonne `mat[:, 1]` donne l'évolution de la température à midi sur l'ensemble des 3 jours !

2. **La vision « Machine Learning » (Échantillons × Caractéristiques)** :
   - Chaque **ligne** est un **échantillon indépendant** (un client, un logement, une session d'appareil).
   - Chaque **colonne** est une **variable mesurée** (*feature* : puissance active, puissance réactive, tension, température).

---

### 💾 1.2 Sous le capot : Comment une matrice 2D est-elle stockée en RAM ?

Rappelez-vous ce que nous avons découvert en **Section 1** : la mémoire vive d'un ordinateur n'a **qu'une seule dimension physique** ! C'est un immense ruban linéaire d'adresses mémoires qui se suivent.

Alors, comment NumPy fait-il pour stocker un tableau 2D de 3 lignes et 4 colonnes ?
- **L'ordre C-Contiguous (Row-Major)** :  
  Par défaut, NumPy range les valeurs **ligne par ligne** les unes derrière les autres :  
  `[ Jour 1 entier (4 valeurs) ][ Jour 2 entier (4 valeurs) ][ Jour 3 entier (4 valeurs) ]`
- **Le secret de `.reshape()` (Zéro copie en mémoire !)** :  
  Lorsque vous écrivez `mat_3x4 = signal_1d.reshape(3, 4)`, NumPy **ne déplace ni ne copie aucun octet** de données en mémoire vive !  
  Il crée simplement un nouvel en-tête (une **Vue**) qui conserve le pointeur vers le même bloc continu, mais adapte les **strides** (le nombre d'octets à sauter pour passer à la ligne suivante).  
  C'est pourquoi `.reshape()` s'exécute en **un temps rigoureusement nul ($O(1)$)**, même sur un tableau de 10 millions de nombres !

---

### 🎯 1.3 Indexation 2D et Découpage (*Slicing*) : La syntaxe `[ligne, colonne]`

Contrairement au Python standard où l'on doit enchaîner deux paires de crochets (`liste[i][j]`, ce qui oblige le processeur à déréférencer deux pointeurs successifs), NumPy utilise une syntaxe compacte avec une **virgule unique** :

```python
mat[i, j]       # Élément précis à la ligne i et colonne j (accès direct en mémoire)
mat[i, :]       # Ligne i complète (le ':' signifie "toutes les colonnes")
mat[:, j]       # Colonne j complète (le ':' signifie "toutes les lignes")
mat[0:2, 1:3]   # Sous-matrice : lignes 0 à 1 incluses, colonnes 1 à 2 incluses
mat[::2, :]     # Une ligne sur deux, toutes les colonnes
```

---

### 💡 1.4 Les deux astuces indispensables en pratique

1. **L'astuce de la dimension automatique `-1`** :  
   Si vous connaissez le nombre de colonnes souhaité mais pas le nombre de lignes (ou inversement), indiquez `-1`. NumPy calculera la dimension manquante à votre place pour que le produit total reste égal à `.size` :
   ```python
   # Exemple avec 12 mesures de température sur 3 jours :
   serie_temp = np.array([18.2, 18.5, 19.1, 20.4, 22.0, 23.5, 24.1, 23.8, 22.1, 20.5, 19.2, 18.6])
   mat_3x4    = serie_temp.reshape(3, -1)   # 3 jours -> NumPy calcule automatiquement 4 créneaux/jour !
   ```

2. **Ajouter un axe avec `np.newaxis` (ou `.reshape(-1, 1)`)** :  
   En Machine Learning (notamment avec **Scikit-Learn**), les modèles refusent catégoriquement les vecteurs 1D de forme `(N,)` et exigent une matrice de forme `(N, 1)`.  
   `np.newaxis` insère une dimension supplémentaire sans dupliquer de mémoire :
   ```python
   vecteur_1d = np.array([10, 20, 30])    # shape : (3,)   <- 1 dimension
   colonne_2d = vecteur_1d[:, np.newaxis] # shape : (3, 1) <- 2 dimensions (matrice colonne)
   ```

Exécutons maintenant le **Programme 2** pour observer la création et l'indexation de cette matrice de températures sur 3 jours :

In [2]:
# Programme 2 — Démonstration 2D et reshape
# 12 relevés de température répartis sur 3 jours (4 créneaux par jour : 6h, 12h, 18h, 24h)
temp_1d = np.array([18.2, 18.5, 19.1, 20.4, 22.0, 23.5, 24.1, 23.8, 22.1, 20.5, 19.2, 18.6])

# Transformation en matrice de 3 jours x 4 créneaux horaires
mat_temp = temp_1d.reshape(3, 4)
print("Matrice 3x4 :\n", mat_temp)
print(f"Shape : {mat_temp.shape}")

# Extraction de la 1ère ligne (Jour 1 complet)
print("\nJour 1 :", mat_temp[0, :])

# Extraction de la 3ème colonne (Créneau 2 pour tous les jours)
print("Créneau 3 pour tous les jours :", mat_temp[:, 2])

Matrice 3x4 :
 [[18.2 18.5 19.1 20.4]
 [22.  23.5 24.1 23.8]
 [22.1 20.5 19.2 18.6]]
Shape : (3, 4)

Jour 1 : [18.2 18.5 19.1 20.4]
Créneau 3 pour tous les jours : [19.1 24.1 19.2]


### ✏️ Exercice 1 — Remodeler un signal continu en matrice

On dispose d'un historique de **24 relevés de température** couvrant **4 journées** (à raison de 6 relevés par jour, soit un toutes les 4 heures), stocké dans le vecteur `releve_24h`.
1. Remodelez ce vecteur en une matrice 2D `mat_jours` de **4 jours** et **6 tranches horaires** par jour (utilisez `.reshape(4, 6)`).
2. Extrayez dans la variable `midi_tous_les_jours` la 4ème tranche horaire (indice colonne `3`) pour l'ensemble des 4 jours.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
mat_jours = releve_24h.reshape(4, 6)
midi_tous_les_jours = mat_jours[:, 3]
```
</details>

In [ ]:
# Programme 3 — Exercice 1
releve_24h = np.array([
    15, 16, 19, 22, 20, 17,  # Jour 1
    14, 15, 18, 24, 21, 16,  # Jour 2
    16, 17, 20, 25, 22, 18,  # Jour 3
    15, 16, 19, 23, 19, 16   # Jour 4
])

# Remplacez les '...' par votre code :
mat_jours = ...
midi_tous_les_jours = ...

In [ ]:
# Programme 4 — Test Exercice 1
check("mat_jours shape", mat_jours.shape if hasattr(mat_jours, 'shape') else None, (4, 6))
check("midi_tous_les_jours", midi_tous_les_jours, np.array([22, 24, 25, 23]))

---
## 2 · Agrégations statistiques et le paramètre `axis`

<div align="center" style="margin: 15px 0 25px 0;">
  <img src="pictures/cartoon_s2_ch2_axis_collapse.jpg" alt="Le grand dilemme d'axis : 0 ou 1" width="85%" style="border-radius: 10px; box-shadow: 0 4px 20px rgba(0,0,0,0.35);" />
  <br>
  <em style="color: #94a3b8; font-size: 0.9em;">🏗️ Le grand piège d'axis enfin démystifié : <code>axis</code> est la dimension qui DISPARAÎT ! La presse <code>axis=0</code> écrase les lignes (somme par colonne), le bulldozer <code>axis=1</code> écrase les colonnes (somme par ligne).</em>
</div>

> 🚜 **La règle d'or du chantier NumPy :**  
> Quand vous tapez `a.mean(axis=k)`, ne pensez pas *« le long de quelle direction je calcule ? »*, mais plutôt :  
> 👉 ***« Quelle dimension dois-je écraser / éliminer ? »***  
> - **`axis=0` (La presse hydraulique)** : s'abat verticalement sur chaque colonne et écrase toutes les lignes (les 3 jours disparaissent). Il ne reste qu'une ligne de totaux (un résultat par colonne / créneau).  
> - **`axis=1` (Le bulldozer latéral)** : pousse horizontalement le long des lignes et écrase toutes les colonnes (les 4 créneaux disparaissent). Il ne reste qu'une colonne de totaux (un résultat par ligne / jour).



Dans la Section 1, nous avons appliqué des fonctions globales comme `signal.mean()` ou `signal.sum()`. Sur un vecteur 1D, le calcul ne pose aucune question d'orientation : toutes les valeurs sont agrégées en un seul nombre scalaire.

Mais sur une matrice 2D, une question fondamentale émerge :  
*« Veut-on la moyenne globale de tout le tableau, la moyenne de chaque jour (par ligne), ou la moyenne de chaque créneau horaire (par colonne) ? »*

C'est ici qu'intervient le paramètre **`axis`**, l'un des outils les plus puissants — mais aussi le piège n°1 des débutants en Data Science !

---

### ⚠️ Le piège mental n°1 : Pourquoi l'intuition nous trompe avec `axis` ?

La confusion universelle chez les débutants s'explique par un raccourci de langage :
- *L'intuition fausse* : « L'axe 0 désigne les lignes. Donc si je veux la moyenne de chaque ligne, je dois écrire `mat.mean(axis=0)` ! » $\rightarrow$ **C'EST EXACTEMENT L'INVERSE !**
- *La réalité mécanique* : Dans NumPy, spécifier `axis=k` signifie que **l'axe $k$ est l'axe qui va être parcouru et écrasé (réduit)** lors du calcul !

> 💡 **La règle d'or infaillible de l'Axe Écrasé :**
> - **`axis = 0` (Direction verticale $\downarrow$)** : Le processeur fait glisser l'opération le long des lignes (les jours).  
>   L'axe des lignes est **écrasé** $\rightarrow$ le résultat possède la forme des colonnes (**un résultat par colonne / créneau**).  
>   *(Exemple : température moyenne au créneau de midi calculée sur l'ensemble des 3 jours).*  
>
> - **`axis = 1` (Direction horizontale $\rightarrow$)** : Le processeur fait glisser l'opération le long des colonnes (les créneaux).  
>   L'axe des colonnes est **écrasé** $\rightarrow$ le résultat possède la forme des lignes (**un résultat par ligne / jour**).  
>   *(Exemple : température moyenne de chaque journée sur ses 4 créneaux).*  

---

### 🌡️ Exemple concret : Analyser les 3 jours de températures (`mat_temp`)

Reprenons exactement notre matrice `mat_temp` issue du **Programme 2** (forme `(3, 4)` : **3 jours** en lignes $\times$ **4 créneaux horaires par jour** en colonnes) :

| Code NumPy | Ce que fait le processeur | Ce que cela représente physiquement | Forme du résultat |
| :--- | :--- | :--- | :--- |
| `mat_temp.mean()` | Moyenne de tous les nombres sans distinction d'axe | La température moyenne globale sur les 3 jours ($20.83\text{ }^\circ\text{C}$). | Scalaire (1 nombre) |
| `mat_temp.mean(axis=0)` | Écrase les 3 jours (direction verticale $\downarrow$) pour chaque créneau | **La journée type sur les 3 jours** : la température moyenne le matin (6h), à midi (12h), l'après-midi (18h) et la nuit (24h). | Vecteur `(4,)` : `[20.77, 20.83, 20.80, 20.93]` |
| `mat_temp.mean(axis=1)` | Écrase les 4 créneaux (direction horizontale $\rightarrow$) pour chaque jour | **La température moyenne quotidienne** : moyenne du Jour 1 ($19.05\text{ }^\circ\text{C}$), du Jour 2 ($23.35\text{ }^\circ\text{C}$) et du Jour 3 ($20.10\text{ }^\circ\text{C}$). | Vecteur `(3,)` : `[19.05, 23.35, 20.10]` |
| `mat_temp.max(axis=1)` | Trouve la valeur maximale sur les 4 créneaux de chaque jour | **Le pic de température de chaque journée** ($20.4\text{ }^\circ\text{C}$ le J1, $24.1\text{ }^\circ\text{C}$ le J2, $22.1\text{ }^\circ\text{C}$ le J3). | Vecteur `(3,)` |
| `mat_temp.argmax(axis=1)` | Trouve l'indice du créneau le plus chaud de chaque jour | **Le créneau du pic quotidien** pour chaque jour (indice `3` le J1, indice `2` le J2, indice `0` le J3). | Vecteur `(3,)` |

> 💡 **Généralisation à grande échelle** : La même règle s'applique à un quartier connecté de 100 maisons mesurées sur 24 heures (`conso_quartier` de forme `(100, 24)`) :  
> - `conso_quartier.mean(axis=0)` réduit les 100 foyers pour donner la **courbe moyenne heure par heure (0h à 23h)** du quartier (vecteur `(24,)`).  
> - `conso_quartier.sum(axis=1)` réduit les 24 heures pour donner la **facture totale de chaque foyer** (vecteur `(100,)`).

---

### 🖼️ Schéma Graphique — Vision 2D, Réduction par Axes et Mémoire RAM

Voici la synthèse visuelle qui réunit la grille 2D de nos 3 jours de températures, le sens des deux axes et l'organisation physique en mémoire :

<div align="center" style="margin: 20px 0;">
  <img src="pictures/numpy_2d_memoire_et_axes.png" alt="Matrices 2D : Vision Géométrique et Réalité en RAM" width="96%" style="border-radius: 10px; box-shadow: 0 4px 20px rgba(0,0,0,0.35);" />
</div>

> 🔍 **Comment lire et décrypter ce schéma technique en 3 volets :**  
> 
> 1. **En haut — La Grille 2D (Vision Modélisation)** :  
>    Notre matrice `mat_temp` de forme `(3, 4)` représente 3 lignes (les 3 jours d'observation) et 4 colonnes (les créneaux 6h, 12h, 18h, 24h). Chaque case est accessible par son couple d'indices `[ligne, colonne]`.
> 
> 2. **Au centre — Le sens des réductions `axis`** :  
>    - **Les flèches bleues descendantes (`axis=0`)** : parcourent verticalement chaque colonne pour fusionner les 3 lignes. Résultat : un vecteur 1D de **4 moyennes** (le profil thermique moyen d'une journée type sur les 3 jours).  
>    - **Les flèches orange horizontales (`axis=1`)** : parcourent horizontalement chaque ligne pour fusionner les 4 colonnes. Résultat : un vecteur 1D de **3 moyennes** (la température moyenne de chaque journée).  
> 
> 3. **En bas — L'alignement en mémoire vive (*C-Contiguous Order*)** :  
>    Regardez la barre de mémoire en bas : la 2D n'existe pas en RAM ! NumPy range d'abord toute la ligne 0 (cases 0 à 3), puis toute la ligne 1 (cases 4 à 7), puis la ligne 2 (cases 8 à 11).  
>    - **Formule d'adresse physique** : Pour trouver la case `[i, j]`, le processeur calcule simplement l'adresse 1D : $\text{Index 1D} = i \times 4 + j$.  
>    - **Zéro surcoût de `ravel()`** : C'est précisément cette contiguïté physique qui permet à `mat_temp.ravel()` d'être instantané ($O(1)$ sans copie mémoire).


---

### 🎯 Trouver la position des extrêmes : `argmin()` et `argmax()`

Alors que `np.max(data)` renvoie la **valeur** maximale, `np.argmax(data)` renvoie l'**indice** (la position) où se trouve ce maximum :
- **Avec `axis=0`** : indice de la ligne maximale pour chaque colonne (résultat de taille `nb_colonnes`).
- **Avec `axis=1`** : indice de la colonne maximale pour chaque ligne (résultat de taille `nb_lignes`).  
  *(Exemple : `mat_temp.argmax(axis=1)` renvoie le créneau horaire du pic pour chacun des 3 jours !)*
- **Sans `axis` (recherche globale)** : par défaut, NumPy aplatit virtuellement la matrice et renvoie un **indice 1D unique à plat** (*flat index*).

---

### 🧶 Zoom technique : Que fait exactement `ravel()` ?

Le verbe anglais *to ravel* signifie **défaire un tricot** ou **dérouler une pelote de laine en un fil unique**.

En NumPy, la méthode **`.ravel()`** prend n'importe quel tableau 2D (ou multi-dimensionnel) et l'**aplatit en un vecteur 1D continu** en lisant les éléments ligne par ligne (ordre C-Order) :

```python
matrice = np.array([
    [10, 20, 30],
    [40, 50, 60]
])  # Forme 2D : (2, 3)

vecteur_1d = matrice.ravel()
# Résultat : array([10, 20, 30, 40, 50, 60])  -> Forme 1D : (6,)
```

#### ⚡ Pourquoi `ravel()` est-il instantané (zéro copie en mémoire) ?
Comme nous l'avons vu sur le schéma ci-dessus, **la 2D n'existe pas physiquement dans la RAM** : les nombres y sont déjà rangés en un bloc 1D continu `[10, 20, 30, 40, 50, 60]` !  
Par conséquent, `.ravel()` ne copie aucun octet de mémoire : il renvoie une simple **Vue (View)** 1D directe sur le même bloc mémoire ($O(1)$).

#### 🛠️ Les deux cas d'usage indispensables en pratique :
1. **Retrouver la valeur du maximum global après `np.argmax()`** :  
   Comme `idx_max = np.argmax(data)` renvoie un indice à plat (ex. `5`), écrire `data.ravel()[idx_max]` donne immédiatement la valeur (ex. `60`) sans avoir à recalculer les indices de ligne et colonne.  
   *(Pour faire l'inverse, c'est-à-dire convertir l'indice `5` en coordonnées 2D `(1, 2)`, on utilise `np.unravel_index(idx_max, data.shape)` : littéralement « détricoter l'indice » !)*
2. **Préparer les données pour Scikit-Learn** :  
   Les modèles de Machine Learning exigent que la variable cible $y$ soit un vecteur 1D de forme `(N,)`. Si votre colonne est sous forme de matrice `(N, 1)`, Scikit-Learn émettra un avertissement vous recommandant d'écrire `y.ravel()`.

> 📌 **`ravel()` vs `flatten()`** :  
> - `a.ravel()` : renvoie une **Vue** sans copie mémoire dès que possible (très rapide, recommandé).  
> - `a.flatten()` : crée **toujours une copie indépendante** en mémoire vive (consomme de la RAM).

Observons ces calculs en direct dans le **Programme 5** :

In [5]:
# Programme 5 — Démonstration de calculs avec axis
data = np.array([
    [10, 20, 30],
    [40, 50, 60]
])

print("Moyenne globale :", data.mean())
print("Moyenne par colonne (axis=0) :", data.mean(axis=0), " -> shape :", data.mean(axis=0).shape)
print("Moyenne par ligne   (axis=1) :", data.mean(axis=1), " -> shape :", data.mean(axis=1).shape)

print("\n--- Recherche d'indices avec argmax ---")
# 1. Par axe (le plus utile sur une matrice)
print("Indice de la ligne max par colonne (axis=0) :", data.argmax(axis=0))
print("Indice de la col max par ligne   (axis=1) :", data.argmax(axis=1))

# 2. Globalement (sans axis) : renvoie l'indice à plat dans le tableau déroulé
idx_max_plat = np.argmax(data)  # Indice dans le tableau aplati [10, 20, 30, 40, 50, 60]
valeur_max = data.ravel()[idx_max_plat]  # .ravel() aplatit la matrice en vecteur 1D
coord_2d = np.unravel_index(idx_max_plat, data.shape)  # Conversion en (ligne, colonne)

print(f"Indice à plat du max : {idx_max_plat} -> valeur = {valeur_max}")
print(f"Coordonnées 2D du max (ligne, col) : {coord_2d}")


Moyenne globale : 35.0
Moyenne par colonne (axis=0) : [25. 35. 45.]  -> shape : (3,)
Moyenne par ligne   (axis=1) : [20. 50.]  -> shape : (2,)

--- Recherche d'indices avec argmax ---
Indice de la ligne max par colonne (axis=0) : [1 1 1]
Indice de la col max par ligne   (axis=1) : [2 2]
Indice à plat du max : 5 -> valeur = 60
Coordonnées 2D du max (ligne, col) : (1, 2)


In [6]:
# Programme 6 — QCM sur axis
qcm(
    "Dans une matrice X de taille (100, 5) représentant 100 maisons et 5 capteurs chacune, que retourne X.mean(axis=0) ?",
    [
        "Un vecteur de 100 valeurs (la moyenne des 5 capteurs pour chaque maison).",
        "Un vecteur de 5 valeurs (la moyenne de chaque capteur à travers les 100 maisons).",
        "Un scalaire unique (la moyenne générale de tout le quartier).",
        "Une matrice carrée (5, 5) de covariance."
    ],
    correct=1,
    explication="axis=0 réduit l'axe des 100 lignes. On moyenne verticalement chaque colonne, ce qui donne 5 moyennes (une par capteur)."
)

### ✏️ Exercice 2 — Statistiques par capteur

Considérons une matrice `releves` de 3 capteurs industriels (lignes) mesurés sur 5 cycles de production (colonnes).
1. Calculez la moyenne de chaque capteur sur ses 5 cycles dans `moyennes_capteurs` (réduction sur l'axe des colonnes, `axis=1`).
2. Calculez la consommation maximale observée sur chaque cycle dans `max_par_cycle` (réduction sur l'axe des capteurs, `axis=0`).

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
moyennes_capteurs = releves.mean(axis=1)
max_par_cycle = releves.max(axis=0)
```
</details>

In [ ]:
# Programme 7 — Exercice 2
releves = np.array([
    [120, 125, 122, 130, 128],   # Capteur A
    [ 80,  82,  85,  79,  81],   # Capteur B
    [210, 215, 205, 220, 218]    # Capteur C
])

# Remplacez les '...' par votre code :
moyennes_capteurs = ...
max_par_cycle = ...

In [ ]:
# Programme 8 — Test Exercice 2
check("moyennes_capteurs", moyennes_capteurs, releves.mean(axis=1))
check("max_par_cycle", max_par_cycle, releves.max(axis=0))

---
## 3 · Le Broadcasting : L'alignement automatique des formes

<div align="center" style="margin: 15px 0 25px 0;">
  <img src="pictures/cartoon_s2_ch3_broadcasting.jpg" alt="Le Broadcasting : Des clones virtuels sans copie" width="85%" style="border-radius: 10px; box-shadow: 0 4px 20px rgba(0,0,0,0.35);" />
  <br>
  <em style="color: #94a3b8; font-size: 0.9em;">✨ La cabine magique du Broadcasting : le petit vecteur <code>(3,)</code> est dupliqué en hologrammes virtuels sur 4 lignes pour matcher la matrice <code>(4, 3)</code> avec 0 octet supplémentaire alloué en RAM !</em>
</div>

> ⚡ **Ce que révèle la cabine de clonage holographique :**  
> - **L'incompatibilité apparente** : En mathématiques pures, soustraire un vecteur de 3 éléments à une matrice de 4 lignes $\times$ 3 colonnes est impossible car les dimensions ne correspondent pas.  
> - **La tentation du débutant** : Créer une boucle `for` sur les 4 lignes, ou copier manuellement le vecteur 4 fois. Sur des millions de lignes en Machine Learning, cela gaspillerait des gigaoctets de mémoire vive.  
> - **Le coup de génie de NumPy (Le Broadcasting)** : NumPy projette des **hologrammes virtuels** en fixant le pas mémoire (*stride*) de la dimension manquante à 0. L'opération s'exécute en une fraction de milliseconde avec **0 octet supplémentaire alloué en RAM** !



Que se passe-t-il si vous voulez soustraire la moyenne de chaque capteur à l'ensemble des mesures ?
La matrice `releves` a une forme `(3, 5)`, et le vecteur des moyennes a une forme `(3,)` ou `(3, 1)`.

Le **Broadcasting** est la capacité de NumPy à étendre virtuellement les petites dimensions sans dupliquer de données en mémoire pour rendre les formes compatibles.

### 📐 Les Règles du Broadcasting
NumPy compare les formes des deux tableaux dimension par dimension, **en partant de la droite** :
Deux dimensions sont compatibles si :
1. Elles sont **égales**, OU
2. L'une d'elles vaut **1** (cette dimension est alors étirée pour matcher l'autre).

### 🎯 Cas d'usage Data Science : Centrer / Normaliser des données (Z-Score)
Normaliser des données signifie soustraire la moyenne et diviser par l'écart-type :
$$Z = \frac{X - \mu}{\sigma}$$
Avec le broadcasting, cette opération complexe s'écrit en une seule ligne limpide !

In [9]:
# Programme 9 — Démonstration du Broadcasting & Standardisation
# Matrice de 4 échantillons (lignes) et 3 caractéristiques (colonnes)
X = np.array([
    [10.0, 100.0, 0.5],
    [12.0, 110.0, 0.6],
    [14.0, 105.0, 0.4],
    [11.0,  95.0, 0.7]
])

# Calcul de la moyenne et de l'écart-type par colonne (axis=0)
mu = X.mean(axis=0)    # shape : (3,)
sigma = X.std(axis=0)  # shape : (3,)

# Standardisation Z-Score grâce au broadcasting :
# X : (4, 3)
# mu :   (3,) -> broadcasté automatiquement sur les 4 lignes !
X_standardise = (X - mu) / sigma

print("Moyennes des colonnes :", np.round(mu, 2))
print("Écarts-types          :", np.round(sigma, 2))
print("\nDonnées standardisées (centrées en 0, écart-type=1) :\n", np.round(X_standardise, 2))
print("\nVérification des nouvelles moyennes :", np.round(X_standardise.mean(axis=0), 6))

Moyennes des colonnes : [ 11.75 102.5    0.55]
Écarts-types          : [1.48 5.59 0.11]

Données standardisées (centrées en 0, écart-type=1) :
 [[-1.18 -0.45 -0.45]
 [ 0.17  1.34  0.45]
 [ 1.52  0.45 -1.34]
 [-0.51 -1.34  1.34]]

Vérification des nouvelles moyennes : [ 0.  0. -0.]


---
## 4 · Nombres Aléatoires avec l'API Moderne (`default_rng`)

<div align="center" style="margin: 15px 0 25px 0;">
  <img src="pictures/cartoon_s2_ch4_default_rng.jpg" alt="Générateur moderne default_rng et graine reproductible" width="85%" style="border-radius: 10px; box-shadow: 0 4px 20px rgba(0,0,0,0.35);" />
  <br>
  <em style="color: #94a3b8; font-size: 0.9em;">🎲 Au cyber-casino de NumPy : le croupier robot <code>default_rng</code> et sa clé dorée <code>seed=42</code> garantissent des simulations 100% reproductibles, pendant que la vieille machine à sous <code>np.random.seed</code> part au musée des antiquités !</em>
</div>

> 🎰 **Ce qu'il faut retenir du cyber-casino NumPy :**  
> - **Pourquoi la vieille machine à sous `np.random.seed` part au musée** : L'ancienne fonction historique de NumPy modifiait un état aléatoire global partagé par tout l'interpréteur Python. Si un module tiers importé dans votre code tirait un nombre aléatoire, tout votre flux de simulation était corrompu !  
> - **Le croupier moderne `default_rng(seed)`** : Introduit avec NumPy 1.17, il instancie un générateur autonome basé sur l'algorithme de pointe **PCG64** (plus rapide, plus uniforme et statistiquement supérieur).  
> - **La clé dorée `seed=42`** : En Data Science et en recherche, fixer la graine aléatoire garantit la **reproductibilité absolue** de vos expériences : n'importe quel collègue réexécutant votre code obtiendra exactement les mêmes résultats au bit près.



Générer des signaux synthétiques, ajouter du bruit réaliste ou initialiser les poids d'un réseau de neurones repose sur le hasard.

> 💡 **Bonne pratique moderne (NumPy $\ge$ 1.17) :**
> Évitez l'ancienne API `np.random.seed(42)` ou `np.random.rand()`.  
> Utilisez le générateur explicite **`np.random.default_rng(seed)`**.  
> Il est plus rapide, possède de meilleures propriétés cryptographiques/statistiques (générateur PCG64) et ne modifie pas l'état global du programme.

### Les distributions incontournables :
- `rng.uniform(low, high, size)` : tirage uniforme continu entre `low` et `high`.
- `rng.normal(loc, scale, size)` : distribution gaussienne (moyenne `loc`, écart-type `scale`). Idéal pour simuler le bruit de mesure.
- `rng.integers(low, high, size)` : entiers aléatoires entre `low` inclus et `high` exclu.
- `rng.choice(a, size=..., replace=...)` : tirage parmi une liste ou un tableau.

In [10]:
# Programme 10 — Démonstration de default_rng
# Création d'un générateur avec graine fixée pour reproductibilité
rng = np.random.default_rng(seed=42)

# 1. Bruit gaussien (moyenne 0, écart-type 2.5 W) sur 5 mesures
bruit = rng.normal(loc=0.0, scale=2.5, size=5)

# 2. Tirage d'entiers simulant des durées de cycle (entre 8 et 15 minutes)
durees = rng.integers(low=8, high=16, size=4)

print("Bruit gaussien (W) :", np.round(bruit, 2))
print("Durées de cycles   :", durees)

Bruit gaussien (W) : [ 0.76 -2.6   1.88  2.35 -4.88]
Durées de cycles   : [12 15 13 14]


---
## 5 · Mini-Projet : Synthèse & Analyse d'un Signal Énergétique avec Lissage

<div align="center" style="margin: 15px 0 25px 0;">
  <img src="pictures/cartoon_s2_ch5_signal_smoothing.jpg" alt="Station de lavage de signal avec np.convolve" width="85%" style="border-radius: 10px; box-shadow: 0 4px 20px rgba(0,0,0,0.35);" />
  <br>
  <em style="color: #94a3b8; font-size: 0.9em;">🧼 La station de lavage du signal : le rouleau compresseur de la moyenne mobile <code>np.convolve(signal, np.ones(21)/21, mode='same')</code> élimine le bruit chaotique et révèle un signal propre et étincelant !</em>
</div>

> 🚿 **Bienvenue dans la station de lavage de signaux réels :**  
> - **Le signal brut pollué** : Dans tout capteur physique (consommation Linky, température, accéléromètre), le signal physique utile est masqué par deux fléaux :  
>   1. Le **bruit thermique haute fréquence** (les oscillations désordonnées de fond).  
>   2. Les **pics d'anomalies parasites** (des surtensions électriques accidentelles).  
> - **Le filtre à moyenne glissante (`np.convolve`)** : Comme un rouleau de lavage microfibre, une fenêtre pondérée de 21 points avance seconde par seconde. Les petites oscillations s'annulent mutuellement et le signal utile en ressort propre, lisse et exploitable pour vos algorithmes de détection !

Mettons en pratique l'ensemble des concepts étudiés à travers une simulation concrète de traitement du signal !

### 🎯 Le Scénario
Nous voulons modéliser et analyser la puissance active $P(t)$ consommée par un appareil industriel sur **1000 secondes** (1 point par seconde) :
1. **Signal nominal** : une oscillation cyclique entre veille (~30 W) et pleine puissance (~150 W).
2. **Bruit physique** : un bruit gaussien de mesure $\pm 5\text{ W}$.
3. **Anomalies** : 3 surtensions parasites accidentelles ($> 200\text{ W}$).

Nous allons utiliser NumPy pour :
- Calculer le taux de marche (% du temps $> 60\text{ W}$).
- Détecter automatiquement les anomalies avec un masque booléen et **localiser leurs indices temporels** avec `np.where(masque)[0]`.
- Lisser le signal avec un filtre à moyenne glissante via la convolution vectorisée `np.convolve`.
- Tracer le résultat avec Matplotlib.

---

### 💡 5.1 Rappel : Pourquoi écrit-on `np.where(masque)[0]` avec un `[0]` ?
- Avec 3 arguments (`np.where(cond, vrai, faux)`), `np.where` remplace des valeurs.
- Avec **1 seul argument** (`np.where(cond)`), `np.where` répond à la question : ***« Où (Where) se trouvent les cases True ? »*** et renvoie leurs **indices (positions)**.
- Comme NumPy gère des tableaux à N dimensions, il renvoie toujours un **tuple de tableaux** (un par dimension).
- Sur notre signal 1D, NumPy renvoie donc un tuple contenant un seul array : `(array([150, 540, 790]),)`.  
- Le suffixe **`[0]`** sert simplement à extraire ce premier array du tuple pour manipuler directement les indices : `indices = np.where(signal > 180.0)[0]`.

---

### 🌊 5.2 Comprendre la Moyenne Mobile et `np.convolve`

Dans le code ci-dessous, vous allez observer ces deux lignes fondamentales en traitement du signal :
```python
fenetre = np.ones(21) / 21
signal_lisse = np.convolve(signal_brut, fenetre, mode='same')
```

### 🖼️ Schéma Graphique — Mécanisme de la Fenêtre Glissante de Convolution

Pour comprendre comment ces deux lignes fonctionnent physiquement, visualisons le passage de la fenêtre glissante sur le signal :

<div align="center" style="margin: 20px 0;">
  <img src="pictures/convolution_moyenne_mobile.png?v=3" alt="Convolution 1D et Fenêtre Glissante (np.convolve)" width="96%" style="border-radius: 10px; box-shadow: 0 4px 20px rgba(0,0,0,0.35);" />
</div>

> 🔍 **Décryptage pas à pas du schéma de convolution :**  
> 
> 1. **La Fenêtre Glissante Mobile (en haut)** :  
>    À chaque instant $t$, une fenêtre de 21 points consécutifs (encadrée en rouge) se positionne sur le signal brut. Elle avance d'un point à chaque étape ($t \rightarrow t+1$).  
>    Pour chaque position, elle effectue le produit élément par élément entre les 21 points du signal et les 21 poids de la fenêtre, puis fait la somme : c'est le **produit de convolution discrète 1D**.
> 
> 2. **Les 3 Clés Techniques (les 3 cartes d'explication au centre)** :  
>    - **`np.ones(K) / K` (Moyenne pondérée)** : `np.ones(21)` crée 21 cases pré-remplies de `1.0`. Diviser par 21 affecte un poids de $1/21 \approx 0.0476$ à chaque point. La somme des 21 poids valant rigoureusement $1.0$, l'échelle physique en Watts est parfaitement conservée (sans ce facteur, la puissance serait multipliée par 21 !).  
>    - **Effet Filtre Passe-Bas (Élimination du bruit)** : Le bruit thermique oscille aléatoirement autour de zéro ($+4\text{ W}, -3.5\text{ W}, \dots$). En moyennant sur 21 secondes, les fluctuations positives et négatives se compensent et s'annulent mutuellement, révélant la vraie forme d'onde de l'appareil.  
>    - **Paramètre `mode='same'` (Alignement temporel)** : Normalement, convoluer 1000 points avec une fenêtre de 21 produit $1000 + 21 - 1 = 1020$ points. L'option `mode='same'` force NumPy à renvoyer un signal lissé de **longueur rigoureusement identique au signal brut ($N = 1000$ points)**. Cela permet de le superposer directement sur l'axe du temps `t` dans Matplotlib sans décalage temporel !  
> 
> 3. **Le Signal Lissé Obtenu (en bas)** :  
>    En sortie, le signal bleu lissé suit fidèlement les cycles de fonctionnement réels de la machine sans le parasitage haute fréquence, ce qui permet de poser des seuils de détection fiables.

---

Exécutons maintenant le **Programme 11** pour observer la détection d'anomalies et le lissage en action :


In [11]:
# Programme 11 — Simulation, traitement du signal et visualisation
# 1. Axe temporel (1000 secondes, 1 mesure/seconde)
t = np.linspace(0, 1000, 1000)

# 2. Signal carré périodique de base (période de 200 s : 80 s ON à 150 W, 120 s OFF à 30 W)
signal_base = np.where((t % 200) < 80, 150.0, 30.0)

# 3. Ajout de bruit gaussien (reproductible)
rng = np.random.default_rng(seed=123)
bruit = rng.normal(loc=0.0, scale=5.0, size=t.size)
signal_brut = signal_base + bruit

# 4. Injection de 3 anomalies ponctuelles à des instants précis
signal_brut[150] = 235.0
signal_brut[540] = 210.0
signal_brut[790] = 245.0

# 5. ANALYSE NUMPY :
# A) Détection des anomalies (> 180 W)
masque_anomalies = signal_brut > 180.0
indices_anomalies = np.where(masque_anomalies)[0]
nb_anomalies = indices_anomalies.size

# B) Taux de fonctionnement (% du temps où P > 60 W hors anomalies)
taux_marche_pct = (np.sum((signal_brut > 60.0) & ~masque_anomalies) / t.size) * 100

# C) Lissage par moyenne mobile sur 21 points avec np.convolve
fenetre = np.ones(21) / 21
signal_lisse = np.convolve(signal_brut, fenetre, mode='same')

print(f"=== Rapport d'Analyse NumPy ===")
print(f"Nombre de points analysés   : {signal_brut.size}")
print(f"Taux de fonctionnement ON   : {taux_marche_pct:.1f} %")
print(f"Anomalies détectées         : {nb_anomalies} pics aux indices {indices_anomalies.tolist()}")
print(f"Puissance max observée      : {signal_brut.max():.1f} W")

# 6. VISUALISATION MATPLOTLIB
plt.figure(figsize=(14, 5))
plt.plot(t, signal_brut, color='#6c757d', alpha=0.5, label='Signal brut bruité')
plt.plot(t, signal_lisse, color='#0d6efd', linewidth=2, label='Signal lissé (moyenne mobile 21 pts)')
plt.scatter(t[indices_anomalies], signal_brut[indices_anomalies], color='#dc3545', s=80, zorder=5, label='Anomalies détectées (>180 W)')

plt.axhline(60, color='#198754', linestyle='--', alpha=0.7, label='Seuil de marche (60 W)')
plt.title("⚡ Signal Électrique Industriel — Traitement et Détection NumPy", fontsize=14, fontweight='bold')
plt.xlabel("Temps (secondes)", fontsize=11)
plt.ylabel("Puissance (Watts)", fontsize=11)
plt.legend(loc='upper right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

=== Rapport d'Analyse NumPy ===
Nombre de points analysés   : 1000
Taux de fonctionnement ON   : 40.1 %
Anomalies détectées         : 3 pics aux indices [150, 540, 790]
Puissance max observée      : 245.0 W


/var/folders/kz/lyx_lm0532g9q8gdt228qdmw0000gn/T/ipykernel_38357/2364176393.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### ✏️ Défi final — Validation de l'analyse

À partir des résultats générés ci-dessus :
1. Affectez à la variable `nb_anom_trouvees` le nombre d'anomalies détectées.
2. Calculez la puissance médiane du signal brut dans `mediane_puissance` avec `np.median()`.
3. Validez vos réponses avec la cellule de test.

<details>
<summary>💡 Cliquez ici pour afficher un indice ou la solution</summary>

```python
nb_anom_trouvees = nb_anomalies
mediane_puissance = np.median(signal_brut)
```
</details>

In [ ]:
# Programme 12 — Défi final
# Remplacez les '...' par votre code :
nb_anom_trouvees = ...
mediane_puissance = ...

In [ ]:
# Programme 13 — Test Défi final
check("nb_anom_trouvees", nb_anom_trouvees, int(nb_anomalies))
check("mediane_puissance", mediane_puissance, float(np.median(signal_brut)), tol=0.1)

---
## 6 · Synthèse Générale & Boîte à Outils (Cheat Sheet)

Félicitations ! Vous maîtrisez désormais l'ensemble des dimensions et des outils avancés de NumPy pour la Data Science et le Machine Learning.

---

### 📋 Mémento des fonctions et modules utilisés

#### 1. Bibliothèque `numpy` (`import numpy as np`) — Matrices 2D, Réductions et Traitement
| Fonction / Attribut | Type | Rôle / Description | Exemple dans le notebook |
| :--- | :--- | :--- | :--- |
| `a.reshape(lignes, cols)` | Méthode | Change la forme 2D du tableau sans copie mémoire ($O(1)$) | `temp_1d.reshape(3, 4)` ou `.reshape(3, -1)` |
| `np.newaxis` | Objet / Index | Insère une nouvelle dimension pour convertir un vecteur 1D en matrice | `signal[:, np.newaxis]  # -> (N, 1)` |
| `a.shape`, `a.ndim`, `a.size` | Attributs | Inspectent respectivement la géométrie, le nombre d'axes et le nombre total d'éléments | `mat_jours.shape  # -> (4, 6)` |
| `a.mean(axis=...)` | Méthode | Moyenne avec paramètre d'axe : `axis=0` (écrase lignes $\rightarrow$ résultat par colonne), `axis=1` (écrase colonnes $\rightarrow$ résultat par ligne) | `mat_temp.mean(axis=1)` |
| `a.max(axis=...)` / `min()` | Méthode | Extrêmes le long de l'axe choisi | `mat_temp.max(axis=0)` |
| `np.argmax(a, axis=...)` | Fonction | Renvoie l'indice de position du maximum le long de l'axe | `mat_temp.argmax(axis=1)` |
| `np.unravel_index(idx, shape)` | Fonction | Convertit un indice à plat (1D) en coordonnées matricielles 2D `(ligne, colonne)` | `np.unravel_index(idx_max, data.shape)` |
| `np.where(cond)[0]` | Fonction | Recherche d'indices : renvoie les positions des `True` (`[0]` extrait le tableau du tuple) | `np.where(signal > 180)[0]` |
| `a.ravel()` | Méthode | Aplatit une matrice en un vecteur 1D continu sans copie mémoire | `data.ravel()[idx_max]` |
| `Broadcasting` | Mécanisme | Alignement automatique des dimensions pour opérer entre matrices et vecteurs | `(X - X.mean(axis=0)) / X.std(axis=0)` |
| `np.convolve(sig, fen, mode='same')` | Fonction | Convolution discrète 1D (moyenne mobile, filtre passe-bas pour lisser le bruit) | `signal_lisse = np.convolve(brut, fenetre, mode='same')` |

#### 2. Génération Aléatoire Moderne (`np.random.default_rng`)
| Méthode | Distribution | Description | Exemple dans le notebook |
| :--- | :--- | :--- | :--- |
| `np.random.default_rng(seed)` | Constructeur | Initialise le générateur pseudo-aléatoire moderne (PCG64) avec une graine reproductible | `rng = np.random.default_rng(123)` |
| `rng.normal(loc, scale, size)` | Gaussienne | Simule un bruit de mesure gaussien centré en `loc` avec un écart-type `scale` | `bruit = rng.normal(0.0, 5.0, size=1000)` |
| `rng.uniform(low, high, size)` | Continue uniforme | Tire des nombres réels équirépartis entre `low` et `high` | `rng.uniform(10.0, 50.0, size=50)` |
| `rng.integers(low, high, size)` | Entière uniforme | Tire des entiers aléatoires entre `low` inclus et `high` exclu | `rng.integers(10, 20, size=4)` |

---

### 🚀 Prochaine étape : Section 3 — Le Perceptron de Rosenblatt !
Vous disposez maintenant de toute la boîte à outils d'algèbre linéaire indispensable pour aborder le Machine Learning :
- Dans la **Section 3**, nous utiliserons nos matrices 2D, le produit matriciel `@`, le broadcasting et les nombres aléatoires pour coder de A à Z notre premier algorithme d'apprentissage automatique : le **Perceptron de Rosenblatt** (1957) !